---
## Configuration

In [ ]:
# ── Google Drive folder ID (from the share link) ──────────────────────────────
DRIVE_FOLDER_ID = '1ktyrAnzQJxG-xOlgfJr70RpJQR2xwB4u'

# ── Where to write all clips (inside your Drive so results are persisted) ─────
# Will be created if it does not exist.
OUTPUT_CLIPS_DIR = '/content/drive/MyDrive/DMD_clips'

# ── Repo ──────────────────────────────────────────────────────────────────────
REPO_DIR = '/content/Driving_Distraction_Detection'
REPO_URL = 'https://github.com/AnnikaUnmuessig/Driving_Distraction_Detection.git'

# ── Behaviour ─────────────────────────────────────────────────────────────────
SKIP_UNCLASSIFIED = False   # set True to drop 'unclassified' clips
DROP_KEYWORDS     = ('hand', 'head', 'face', 'gaze', 'eye', 'eyes', 'mirror')

print('Configuration loaded ✓')

---
## 1 · Mount Drive

In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')

# Authenticate so we can call the Drive API
auth.authenticate_user()
print('Drive mounted and authenticated ✓')

In [ ]:
from googleapiclient.discovery import build
from pathlib import Path

_drive_svc = build('drive', 'v3')


def _get_meta(file_id: str) -> dict:
    return _drive_svc.files().get(
        fileId=file_id,
        fields='id,name,parents,driveId',
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()


def resolve_drive_path(folder_id: str) -> Path:
    """Walk parent chain to build the mounted filesystem path."""
    parts = []
    current_id = folder_id
    visited = set()

    while current_id and current_id not in visited:
        visited.add(current_id)
        meta = _get_meta(current_id)
        parts.append(meta['name'])
        parents = meta.get('parents', [])
        if not parents:
            break
        current_id = parents[0]

    parts.reverse()

    # Determine mount root: Shared Drive vs My Drive
    first_meta = _get_meta(folder_id)
    drive_id   = first_meta.get('driveId')
    if drive_id:
        drive_name = _drive_svc.drives().get(driveId=drive_id).execute()['name']
        # parts[0] is the drive name itself; skip it and prepend Shareddrives/<name>
        tail = parts[1:]  # everything below the drive root
        return Path('/content/drive/Shareddrives') / drive_name / Path(*tail) if tail else Path('/content/drive/Shareddrives') / drive_name
    else:
        # My Drive: parts[0] == 'My Drive'
        tail = parts[1:]
        return Path('/content/drive/MyDrive') / Path(*tail) if tail else Path('/content/drive/MyDrive')


DATA_ROOT = resolve_drive_path(DRIVE_FOLDER_ID)
print(f'Resolved folder path: {DATA_ROOT}')
assert DATA_ROOT.is_dir(), f'Path does not exist on the mounted filesystem: {DATA_ROOT}'

---
## 2 · Install dependencies & clone repo

In [ ]:
"""
%pip install -q opencv-python tqdm

import sys, os
repo_path = Path(REPO_DIR)
if repo_path.exists():
    print('Repo exists – pulling …')
    os.system(f'git -C "{REPO_DIR}" pull --ff-only')
else:
    print('Cloning repo …')
    os.system(f'git clone "{REPO_URL}" "{REPO_DIR}"')

scripts_path = str(repo_path / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

print('Dependencies ready ✓')
"""

---
## 3 · Imports & class map

In [ ]:
from __future__ import annotations

import json, re
from typing import Any

import cv2
from tqdm.notebook import tqdm

from clean_dmd_body_actions import clean_dmd_json  # from cloned repo

DMD_BODY_ACTION_MAP: dict[str, int] = {
    'safe_driving':         0,
    'texting_right':        1,
    'phonecall_right':      2,
    'texting_left':         3,
    'phonecall_left':       4,
    'radio':                5,
    'drinking':             6,
    'reach_side':           7,
    'hair_and_makeup':      8,
    'talking_to_passenger': 9,
    'reach_backseat':       10,
    'change_gear':          11,
    'stand_still_waiting':  12,
    'unclassified':         13,
}

print(f'Imports OK ✓  ({len(DMD_BODY_ACTION_MAP)} classes)')

---
## 4 · Helper functions

In [ ]:
_VIDEO_STEM_RE = re.compile(r'^(.+)_rgb_body$')


def find_leaf_folders(root: Path) -> list[Path]:
    """
    Recursively walk *root* and return every folder that contains at least
    one '*_rgb_body.mp4' file (= a 'base case' ready for processing).
    """
    leaves: list[Path] = []
    for folder in sorted(root.rglob('*')):
        if not folder.is_dir():
            continue
        # A leaf folder contains at least one body-camera video
        videos = [f for f in folder.iterdir()
                  if f.suffix == '.mp4' and _VIDEO_STEM_RE.match(f.stem)]
        if videos:
            leaves.append(folder)
    return leaves


def find_pairs_in_folder(folder: Path) -> list[tuple[Path, Path]]:
    """
    Inside a leaf folder, pair each '*_rgb_body.mp4' with its
    '*_rgb_ann_distraction_body_only.json' (already cleaned).
    """
    pairs: list[tuple[Path, Path]] = []
    for mp4 in sorted(folder.glob('*.mp4')):
        m = _VIDEO_STEM_RE.match(mp4.stem)
        if not m:
            continue
        prefix    = m.group(1)
        json_path = folder / f'{prefix}_rgb_ann_distraction_body_only.json'
        if json_path.exists():
            pairs.append((mp4, json_path))
        else:
            print(f'  [WARN] No body_only JSON for {mp4.name}')
    return pairs


def load_actions(json_path: Path) -> list[dict[str, Any]]:
    with json_path.open('r', encoding='utf-8') as fh:
        payload = json.load(fh)
    actions_raw = payload.get('openlabel', {}).get('actions', {})
    actions: list[dict[str, Any]] = []
    for action_id, action_data in actions_raw.items():
        atype = str(action_data.get('type', 'unclassified')).lower()
        for iv in action_data.get('frame_intervals', []):
            actions.append({
                'id': action_id, 'type': atype,
                'frame_start': int(iv['frame_start']),
                'frame_end':   int(iv['frame_end']),
            })
    actions.sort(key=lambda a: (a['frame_start'], a['frame_end']))
    return actions


def write_clip(cap, out_path: Path, fs: int, fe: int,
               fps: float, w: int, h: int) -> bool:
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    cap.set(cv2.CAP_PROP_POS_FRAMES, fs)
    written = 0
    for _ in range(fe - fs + 1):
        ret, frame = cap.read()
        if not ret:
            break
        writer.write(frame)
        written += 1
    writer.release()
    if written == 0:
        out_path.unlink(missing_ok=True)
        return False
    return True


def sanitize(s: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', s)


print('Helpers defined ✓')

---
## 5 · Run: clean → cut → collect

For every leaf folder the pipeline:
1. Cleans each raw annotation JSON → `*_body_only.json` (skipped if exists).
2. Cuts the corresponding video into clips → `OUTPUT_CLIPS_DIR/<class_name>/`.

In [ ]:
clips_dir = Path(OUTPUT_CLIPS_DIR)
clips_dir.mkdir(parents=True, exist_ok=True)
for cls in DMD_BODY_ACTION_MAP:
    (clips_dir / cls).mkdir(exist_ok=True)

print(f'Output dir: {clips_dir}')
print('Class subfolders created ✓')

# Find all leaf folders
leaf_folders = find_leaf_folders(DATA_ROOT)
print(f'\nLeaf folders found: {len(leaf_folders)}')
for lf in leaf_folders:
    print(f'  {lf.relative_to(DATA_ROOT)}')

In [ ]:
drop_kws    = tuple(k.lower() for k in DROP_KEYWORDS)
label_lines: list[str]  = []
stats: dict[str, int]   = {k: 0 for k in DMD_BODY_ACTION_MAP}

for leaf in tqdm(leaf_folders, desc='Leaf folders', unit='folder'):
    print(f'\n {leaf.relative_to(DATA_ROOT)}')

    # ── Step A: clean raw annotation JSONs in this folder ────────────────────
    raw_jsons = [
        p for p in leaf.glob('*.json')
        if 'ann_distraction' in p.stem and '_body_only' not in p.stem
    ]
    for raw_json in raw_jsons:
        out_json = raw_json.with_name(f'{raw_json.stem}_body_only.json')
        if out_json.exists():
            print(f'  [SKIP clean] {out_json.name}')
            continue
        result   = clean_dmd_json(raw_json, out_json, drop_kws)
        n_act    = len(result.get('openlabel', {}).get('actions', {}))
        print(f'  ✓ cleaned {out_json.name}  ({n_act} body actions)')

    # ── Step B: find video–annotation pairs ──────────────────────────────────
    pairs = find_pairs_in_folder(leaf)
    if not pairs:
        print('  [WARN] No video-annotation pairs found, skipping.')
        continue

    # ── Step C: cut each video ───────────────────────────────────────────────
    for video_path, json_path in pairs:
        actions = load_actions(json_path)
        if not actions:
            print(f'  [WARN] No actions in {json_path.name}')
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f'  [ERROR] Cannot open {video_path.name}')
            continue

        fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f'{video_path.name}  [{total} frames | {fps:.1f} fps | {w}x{h}]')

        for idx, action in enumerate(
            tqdm(actions, desc='    clips', unit='clip', leave=False)
        ):
            label_str = action['type']
            class_id  = DMD_BODY_ACTION_MAP.get(label_str, 13)  # 13 = unclassified

            if SKIP_UNCLASSIFIED and class_id == 13:
                continue

            fs = max(0, action['frame_start'])
            fe = min(total - 1, action['frame_end'])
            if fs > fe:
                continue

            clip_name = f'{video_path.stem}_clip{idx:04d}_{sanitize(label_str)}.mp4'
            out_path  = clips_dir / label_str / clip_name

            if out_path.exists():
                # Re-running: skip write but still register label
                label_lines.append(f'{label_str}/{clip_name} {class_id}')
                stats[label_str] += 1
                continue

            if write_clip(cap, out_path, fs, fe, fps, w, h):
                label_lines.append(f'{label_str}/{clip_name} {class_id}')
                stats[label_str] += 1

        cap.release()

print(f'\nDone — {len(label_lines)} clips total')

---
## 6 · Write labels.txt

In [ ]:
labels_path = clips_dir / 'labels.txt'
with labels_path.open('w', encoding='utf-8') as fh:
    fh.write('\n'.join(label_lines))
    if label_lines:
        fh.write('\n')

print(f'Written: {labels_path}  ({len(label_lines)} lines)')
print('\nSample (first 10 lines):')
for line in label_lines[:10]:
    print(' ', line)

---
## Summary

In [ ]:
from pathlib import Path

# Initialize a dictionary to store frame counts per class
frame_counts_per_class = {k: 0 for k in DMD_BODY_ACTION_MAP}

# Define the output file path in the same directory as the clips
summary_output_dir = Path(OUTPUT_CLIPS_DIR)
summary_output_dir.mkdir(parents=True, exist_ok=True) # Ensure the directory exists
output_file_path = summary_output_dir / "class_summary.txt"

print(f"Generating class summary file: {output_file_path}")

# Re-iterate through the data to collect frame counts
for leaf in leaf_folders:
    pairs = find_pairs_in_folder(leaf)
    for video_path, json_path in pairs:
        # Load actions for this JSON file
        actions = load_actions(json_path)

        for action in actions:
            label_str = action['type']
            # Apply the same logic for determining the class label as in the video clipping step
            label_str = label_str if label_str in DMD_BODY_ACTION_MAP else 'unclassified'
            class_id = DMD_BODY_ACTION_MAP[label_str]

            # Skip unclassified if configured to do so (consistent with clip generation)
            if SKIP_UNCLASSIFIED and class_id == DMD_BODY_ACTION_MAP['unclassified']:
                continue

            fs = max(0, action['frame_start'])
            fe = action['frame_end'] # The frame count is based on the annotation interval.

            if fs <= fe: # Only count if start frame is not after end frame
                frame_count = fe - fs + 1
                frame_counts_per_class[label_str] += frame_count

# Now, write the summary to a file
with open(output_file_path, 'w', encoding='utf-8') as f:
    f.write("Class Name,Number of Clips,Number of Frames\n") # Header
    # Iterate through classes sorted by their ID for consistent output
    for class_name, class_id in sorted(DMD_BODY_ACTION_MAP.items(), key=lambda item: item[1]):
        if SKIP_UNCLASSIFIED and class_id == DMD_BODY_ACTION_MAP['unclassified']:
            continue
        
        num_clips = stats.get(class_name, 0) # Use the pre-computed 'stats' for clip counts
        num_frames = frame_counts_per_class.get(class_name, 0)
        f.write(f"{class_name},{num_clips},{num_frames}\n")

print(f"Summary successfully written to {output_file_path}")

Huggingface Upload

In [ ]:
import os
!pip install huggingface_hub -q

from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login, HfApi

login(token="")  # token con permessi write
api = HfApi()

# Carica tutto mantenendo la struttura classi/video.mp4
# Utilizza upload_large_folder per la gestione ottimale di grandi cartelle
api.upload_large_folder(
    folder_path="/content/drive/MyDrive/DMD_clips/",
    repo_id="endoard/distraction_dataset", 
    repo_type="dataset"
)